# A3-M3-2 · 메타데이터 필터와 수정
교안의 같은 번호 단원과 함께 봅니다. 노트북은 01 → 02 → 03 순서로 실행합니다.

## 02-1. 준비 — 저장된 컬렉션 다시 열기

In [ ]:
import pandas as pd
import chromadb
from emb import embed

client = chromadb.PersistentClient(path="chroma_db")
col = client.get_collection("shop_faq")                     # 01에서 만든 컬렉션 다시 열기
print("저장된 개수:", col.count())

def search(question, k=3, where=None, where_document=None):
    r = col.query(query_embeddings=embed([question], kind="query"), n_results=k,
                  where=where, where_document=where_document)
    return pd.DataFrame({
        "faq_id": [int(i) for i in r["ids"][0]],
        "score": [round(1 - d, 3) for d in r["distances"][0]],
        "category": [m["category"] for m in r["metadatas"][0]],
        "question": [m["question"] for m in r["metadatas"][0]],
    })

## 02-2. where — 카테고리 하나로 거르기

In [ ]:
q = "돈 언제 돌려줘요?"
print("[필터 없음]")
print(search(q).to_string(index=False))
print("\n[환불만]  where={'category': '환불'}")
print(search(q, where={"category": "환불"}).to_string(index=False))

## 02-3. where 연산자 — 여러 개, 빼고, 숫자 비교

In [ ]:
q = "돈 언제 돌려줘요?"
print("[환불 또는 결제]")
print(search(q, where={"category": {"$in": ["환불", "결제"]}}).to_string(index=False))
print("\n[회원 빼고]")
print(search(q, where={"category": {"$ne": "회원"}}).to_string(index=False))

## 02-4. where_document — 원문에 특정 단어가 있는 것만

In [ ]:
q = "어디서 확인해요?"
print("[필터 없음]")
print(search(q).to_string(index=False))
print("\n[원문에 '마이페이지'가 들어간 것만]")
print(search(q, where_document={"$contains": "마이페이지"}).to_string(index=False))

## 03-1. update · upsert · delete — 데이터 고치기

In [ ]:
col.update(ids=["601"], metadatas=[{"category": "회원", "question": "비밀번호를 잊어버렸어요.", "updated": "2026-10"}])
print("update 후:", col.get(ids=["601"])["metadatas"])

col.upsert(ids=["999"], documents=["임시 FAQ 테스트 문서"], metadatas=[{"category": "테스트", "question": "임시"}],
           embeddings=embed(["임시 FAQ 테스트 문서"]))
print("upsert 후 개수:", col.count())

col.delete(ids=["999"])
print("delete 후 개수:", col.count())

## 03-2. 자주 하는 실수 두 가지

In [ ]:
try:
    col.add(ids=[101], documents=["x"], embeddings=embed(["x"]))       # id를 숫자로
except Exception as e:
    print("① id가 숫자:", type(e).__name__, "-", str(e)[:80])

n = col.count()
col.add(ids=["101"], documents=["다른 내용"], metadatas=[{"category": "배송", "question": "?"}], embeddings=embed(["다른 내용"]))
print("② 이미 있는 id로 add:", n, "→", col.count(), "| 101 원문:", col.get(ids=["101"])["documents"][0][:20], "...")